In [ ]:
from pathlib import Path
import sys
import importlib
import polars as pl 

pl.Config.set_tbl_rows(1000)   # allow many rows
pl.Config.set_tbl_cols(100)    # allow many columns
pl.Config.set_tbl_width_chars(200)


current = Path.cwd()

while current.name != "shared-notebooks":
    if current.parent == current:
        raise RuntimeError("Could not locate shared-notebooks directory")
    current = current.parent

utils_path = current / "common_utils" / "python"
if str(utils_path) not in sys.path:
    sys.path.append(str(utils_path))

import load_flight_data
importlib.reload(load_flight_data)

lf = load_flight_data.load_flight_data(file_name = 'flights_canonical_2019.parquet')

NY_MARKET = ["JFK", "LGA", "EWR"]

lf_chain = (
    lf
    .filter(
        pl.col("tail_number").is_not_null() &
        pl.col("dep_ts_actual_utc").is_not_null()
    )
    .sort(["tail_number", "dep_ts_actual_utc"])
    .with_columns([
        pl.col("origin").shift(1).over("tail_number").alias("prev1_origin"),
        pl.col("dest").shift(1).over("tail_number").alias("prev1_dest"),
        pl.col("arr_ts_actual_utc").shift(1).over("tail_number").alias("prev1_arr_ts_utc"),
        pl.col("flight_id").shift(1).over("tail_number").alias("prev1_flight_id"),

        pl.col("origin").shift(2).over("tail_number").alias("prev2_origin"),
        pl.col("dest").shift(2).over("tail_number").alias("prev2_dest"),
        pl.col("arr_ts_actual_utc").shift(2).over("tail_number").alias("prev2_arr_ts_utc"),
        pl.col("flight_id").shift(2).over("tail_number").alias("prev2_flight_id"),

        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(1).over("tail_number")
        ).dt.total_minutes().alias("prev1_turnaround_minutes"),

        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(2).over("tail_number")
        ).dt.total_minutes().alias("time_since_prev2_arrival_minutes"),
    ])
)

df = (
    lf_chain
    # .select([
    #     "dep_delay",
    #     "flight_date",
    #     "origin",
    #     "dest"
    # ])
    .filter(pl.col("dep_delay").is_not_null())
    # .head(100)
    .collect()
)

df.height
